In [ ]:
# =========================================================
# Part 0: Import Required Libraries
# =========================================================

# Numerical and data handling
import numpy as np
import pandas as pd

# Load MATLAB / Octave data
from scipy.io import loadmat

# Visualization
import matplotlib.pyplot as plt

# Data preprocessing and model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression, Ridge

# Feature engineering
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Model pipeline
from sklearn.pipeline import Pipeline

# Evaluation metrics for error analysis
from sklearn.metrics import (
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    classification_report
)

# Utility
from sklearn.model_selection import learning_curve

#### =============================================================
#### Part 1: Model Selection and Bias/Variance
#### Dataset: WaterFlow_Data.mat
#### =============================================================

In the part, you will implement regularized linear regression to predict the amount of water flowing out of a dam using the change of water level in a reservoir. In addition, you will go through some diagnostics of debugging learning algorithms and examine the effects of bias vs. variance.

Ref: Machine Learning by Andrew Ng: www.coursera.org/

In [ ]:
# -----------------------------
# Step 1: Load and Inspect the Dataset
# -----------------------------

# Load the MATLAB / Octave data file
data = loadmat("Data/WaterFlow_Data.mat")

# Display available variables in the dataset
data.keys()

In [ ]:
# -----------------------------
# Step 2: Extract Training, Cross-Validation, and Test Sets
# -----------------------------

# Training set
X_train = data["X"]
y_train = data["y"]

# Cross-validation set
X_cv = data["Xval"]
y_cv = data["yval"]

# Test set
X_test = data["Xtest"]
y_test = data["ytest"]

In [ ]:
# -----------------------------
# Step 3: Adjust Target Variable Shape
# -----------------------------

# Convert column vectors (m, 1), which are 2D arrays, to 1D arrays (m,)
y_train = y_train.ravel()
y_cv = y_cv.ravel()
y_test = y_test.ravel()

# .ravel() flattens a column vector into a 1-D array
# scikit-learn expects target vectors (y) to be 1D arrays (m,), not column vectors (m, 1)
# In scikit-learn, X is always 2D array, and y is always 1D array

In [ ]:
# -----------------------------
# Step 4: Sanity Check the Dataset
# -----------------------------

print("Training set:", X_train.shape, y_train.shape)
print("Cross-validation set:", X_cv.shape, y_cv.shape)
print("Test set:", X_test.shape, y_test.shape)

In [ ]:
# -----------------------------
# Step 5: Visualize the Training Data
# -----------------------------

plt.figure(figsize = (6, 4))
plt.plot(
    X_train,
    y_train,
    'ro',
    ms = 10,      # marker size
    mec = 'k',    # marker edge color (k = black)
    mew = 1       # marker edge width
)
plt.xlabel("Water Flow")
plt.ylabel("Change in Water Level")
plt.title("Training Data")
plt.grid(True)
plt.show()

In [ ]:
# -----------------------------
# Step 6: Train Unregularized Linear Regression
# -----------------------------

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

In [ ]:
# -----------------------------
# Step 7: Plot Linear Regression Fit
# -----------------------------

# Generate points for plotting the regression line
X_plot = np.linspace(X_train.min(), X_train.max(), 100).reshape(-1, 1)
y_plot = lin_reg.predict(X_plot)

plt.figure(figsize = (6, 4))

# Plot training data
plt.plot(
    X_train,
    y_train,
    'ro',
    ms = 10,      # marker size
    mec = 'k',    # marker edge color
    mew = 1,
    label = "Training data"
)

# Plot linear regression hypothesis
plt.plot(
    X_plot,
    y_plot,
    'b-',
    linewidth = 2,
    label = "Linear regression"
)

plt.xlabel("Change in Water Level")
plt.ylabel("Water Flow")
plt.title("Unregularized Linear Regression")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# -----------------------------
# Step 8: Define the Cost Function
# -----------------------------

def compute_cost(y_true, y_pred):
    """
    Compute the cost function:
    J = (1 / (2m)) * sum((h(x) - y)^2)
    """
    return mean_squared_error(y_true, y_pred) / 2

# Linear regression cost function J in the lecture slide is equivalent to MSE/2

In [ ]:
# -----------------------------
# Step 9: Compute Learning Curve Data
# -----------------------------

m_train = X_train.shape[0]

train_errors = []
cv_errors = []

for m in range(1, m_train + 1):
    # Train model using first m examples
    lin_reg.fit(X_train[:m], y_train[:m])
    
    # Training error (from first m examples)
    y_train_pred = lin_reg.predict(X_train[:m])
    train_errors.append(compute_cost(y_train[:m], y_train_pred))
    
    # Cross-validation error (from entire CV set)
    y_cv_pred = lin_reg.predict(X_cv)
    cv_errors.append(compute_cost(y_cv, y_cv_pred))

In [ ]:
# -----------------------------
# Step 10: Plot Learning Curves (Underfitting Diagnosis)
# -----------------------------

plt.figure(figsize = (6, 4))
plt.plot(range(1, m_train + 1), train_errors, "b-", label = "Training error")
plt.plot(range(1, m_train + 1), cv_errors, "r-", label = "Cross-validation error")
plt.xlabel("Number of Training Examples")
plt.ylabel("Cost J")
plt.title("Learning Curve (Unregularized Linear Regression)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# -----------------------------
# Step 11: Map Feature to Polynomial Features (Degree = 8)
# -----------------------------

# Degree of polynomial
degree = 8

# Create polynomial features (exclude bias term)
poly = PolynomialFeatures(degree = degree, include_bias = False)

# Map features
X_poly_train = poly.fit_transform(X_train)
X_poly_cv = poly.transform(X_cv)
X_poly_test = poly.transform(X_test)

# .fit() learns parameters from the data, but does not return transformed data
# .transform() applies a previously learned transformation
# .fit_transform() = .fit() + .transform()
# Anything with .fit() must use only training data
# We call .fit() only on the training set to prevent data leakage and ensure fair evaluation on validation and test sets

In [ ]:
# -----------------------------
# Step 12: Feature Normalization
# -----------------------------

# Normalize features to have zero mean and unit variance
# i.e., x_norm = (x - μ)/σ
scaler = StandardScaler()

X_poly_train_norm = scaler.fit_transform(X_poly_train)
X_poly_cv_norm = scaler.transform(X_poly_cv)
X_poly_test_norm = scaler.transform(X_poly_test)

# scaler.fit_transform(X_train) -> learn mean and standard deviation from training data
# scaler.transform(X_poly_cv), scaler.transform(X_poly_test) -> apply the same scaling everywhere else

In [ ]:
# -----------------------------
# Step 13: Train Regularized Linear Regression (Single Lambda Example)
# -----------------------------

# Example regularization parameter
lambda_example = 1.0

ridge_reg = Ridge(alpha = lambda_example)
ridge_reg.fit(X_poly_train_norm, y_train)

# We do not include a bias term in X because scikit-learn automatically adds an intercept,
# and it is not regularized

In [ ]:
# -----------------------------
# Step 14: Plot Polynomial Regression Fit
# -----------------------------

# Generate points for smooth curve plotting
X_plot = np.linspace(X_train.min(), X_train.max(), 100).reshape(-1, 1)
X_plot_poly = poly.transform(X_plot)
X_plot_poly_norm = scaler.transform(X_plot_poly)

y_plot = ridge_reg.predict(X_plot_poly_norm)

plt.figure(figsize = (6, 4))

# Training data
plt.plot(
    X_train,
    y_train,
    'ro',
    ms = 10,
    mec = 'k',
    mew = 1,
    label = "Training data"
)

# Polynomial regression curve
plt.plot(
    X_plot,
    y_plot,
    'b-',
    linewidth = 2,
    label = f"Polynomial regression (λ = {lambda_example})"
)

plt.xlabel("Change in Water Level")
plt.ylabel("Water Flow")
plt.title("Polynomial Regression with Regularization")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# -----------------------------
# Step 15: Select Best Lambda Using Cross-Validation
# -----------------------------

# List of regularization parameters to evaluate
lambda_values = [0, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]

train_errors = []
cv_errors = []

for lam in lambda_values:
    model = Ridge(alpha = lam)
    model.fit(X_poly_train_norm, y_train)
    
    # use compute_cost function defined in Step 8
    # Training error (without regularization term)
    y_train_pred = model.predict(X_poly_train_norm)
    train_errors.append(compute_cost(y_train, y_train_pred))
    
    # Cross-validation error (without regularization term)
    y_cv_pred = model.predict(X_poly_cv_norm)
    cv_errors.append(compute_cost(y_cv, y_cv_pred))

# Display results in table format
print("{:<10} {:<20} {:<20}".format("Lambda", "Training Error", "CV Error"))
print("-" * 50)

for lam, tr_err, cv_err in zip(lambda_values, train_errors, cv_errors):
    print("{:<10} {:<20.6f} {:<20.6f}".format(lam, tr_err, cv_err))

In [ ]:
# -----------------------------
# Step 16: Plot Validation Curve (Lambda Selection - Explicit Lambda Values)
# -----------------------------

# Create index positions for each lambda
x_pos = np.arange(len(lambda_values))

plt.figure(figsize = (7, 4))

# Plot training error
plt.plot(
    x_pos,
    train_errors,
    'b-o',    # blue solid line with circular markers
    linewidth = 2,
    label = "Training error"
)

# Plot cross-validation error
plt.plot(
    x_pos,
    cv_errors,
    'r-o',    # red solid line with circular markers
    linewidth = 2,
    label = "Cross-validation error"
)

# Set x-axis ticks to actual lambda values
plt.xticks(x_pos, lambda_values)

plt.xlabel("Lambda (Regularization Parameter)")
plt.ylabel("Cost J")
plt.title("Selecting Lambda Using Cross-Validation")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# -----------------------------
# Step 17: Select the Best Lambda
# -----------------------------

# Convert lists to NumPy arrays for convenience
train_errors = np.array(train_errors)
cv_errors = np.array(cv_errors)

# Select lambda with minimum cross-validation error
best_lambda_index = np.argmin(cv_errors)
best_lambda = lambda_values[best_lambda_index]

print(f"Best lambda selected using cross-validation: {best_lambda}")

In [ ]:
# -----------------------------
# Step 18: Train Final Model Using Best Lambda
# -----------------------------

final_model = Ridge(alpha = best_lambda)
final_model.fit(X_poly_train_norm, y_train)

In [ ]:
# -----------------------------
# Step 19: Compute Test Set Error
# -----------------------------

# Predict on test set
y_test_pred = final_model.predict(X_poly_test_norm)

# Compute test error (without regularization term)
test_error = compute_cost(y_test, y_test_pred)

print(f"Test set error (J): {test_error:.6f}")

In [ ]:
# -----------------------------
# Step 20: Plot Learning Curve with Regularization (using best lambda)
# -----------------------------

m_train = X_train.shape[0]

train_errors_reg = []
cv_errors_reg = []

for m in range(1, m_train + 1):
    # Train model on first m examples
    model = Ridge(alpha = best_lambda)
    model.fit(X_poly_train_norm[:m], y_train[:m])
    
    # Training error (without regularization term)
    y_train_pred = model.predict(X_poly_train_norm[:m])
    train_errors_reg.append(compute_cost(y_train[:m], y_train_pred))
    
    # Cross-validation error (without regularization term)
    y_cv_pred = model.predict(X_poly_cv_norm)
    cv_errors_reg.append(compute_cost(y_cv, y_cv_pred))


# Plot learning curves
plt.figure(figsize = (6, 4))

plt.plot(
    range(1, m_train + 1),
    train_errors_reg,
    'b-o',
    label = "Training error"
)

plt.plot(
    range(1, m_train + 1),
    cv_errors_reg,
    'r-o',
    label = "Cross-validation error"
)

plt.xlabel("Number of Training Examples")
plt.ylabel("Cost J")
plt.title("Learning Curve (Polynomial Regression with Regularization)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# -----------------------------
# Step 21: Plot Final Hypothesis (using final model with best lambda from step 18)
# -----------------------------

# Generate points for smooth curve plotting
X_plot = np.linspace(X_train.min(), X_train.max(), 100).reshape(-1, 1)
X_plot_poly = poly.transform(X_plot)
X_plot_poly_norm = scaler.transform(X_plot_poly)

y_plot = final_model.predict(X_plot_poly_norm) 

plt.figure(figsize = (6, 4))

# Training data
plt.plot(
    X_train,
    y_train,
    'ro',
    ms = 10,
    mec = 'k',
    mew = 1,
    label = "Training data"
)

# Final hypothesis
plt.plot(
    X_plot,
    y_plot,
    'b-',
    linewidth = 2,
    label = f"Final model (λ = {best_lambda})"
)

plt.xlabel("Change in Water Level")
plt.ylabel("Water Flow")
plt.title("Final Polynomial Regression Model")
plt.legend()
plt.grid(True)
plt.show()

#### =============================================================
#### Part 2: Error Analysis (Binary Classification)
#### Dataset: BreastCancer_Data.csv
#### =============================================================

Breast cancer is the most common cancer amongst women in the world. It accounts for 25% of all cancer cases, and affected over 2.1 Million people in 2015 alone. It starts when cells in the breast begin to grow out of control. These cells usually form tumors that can be seen via X-ray or felt as lumps in the breast area.

The key challenges against it’s detection is how to classify tumors into malignant (cancerous) or benign (non cancerous). 

Ref: Kaggle Dataset https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset

In [ ]:
# -----------------------------
# Step 1: Load the Dataset
# -----------------------------

# Load binary classification dataset
# Target column: 'diagnosis' (M = Malignant, B = Benign)
df = pd.read_csv("Data/BreastCancer_Data.csv")

# Display first few rows to understand the data structure
df.head()

In [ ]:
# -----------------------------
# Step 2: Separate Features and Target
# -----------------------------

# Features: all numeric feature columns (30 features)
X = df.drop(columns = ["diagnosis"])

# Target variable
y = df["diagnosis"]

# Check class labels
print("Original class labels:", y.unique())

In [ ]:
# -----------------------------
# Step 2.1: Examine Class Distribution (Imbalanced Dataset)
# -----------------------------

# Count number of samples in each class
class_counts = y.value_counts()

print("Class distribution:")
print(class_counts)

# .value_counts() counts how many samples belong to each class in a dataset

# Compute class proportions
class_proportions = y.value_counts(normalize = True)

print("\nClass proportions:")
print(class_proportions)

# .value_counts(normalize = True) returns proportions instead of raw counts

In [ ]:
# -----------------------------
# Step 2.2: Visualize Class Distribution
# -----------------------------

plt.figure(figsize = (5, 4))
class_counts.plot(kind = "bar", color =["steelblue", "salmon"])
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.title("Class Distribution (Benign vs Malignant)")
plt.grid(axis = "y")
plt.show()

In [ ]:
# -----------------------------
# Step 3: Encode Target Variable
# -----------------------------

# Encode target variable:
# Malignant (M) -> 1  (positive class)
# Benign (B)    -> 0  (negative class)
y_encoded = y.map({"M": 1, "B": 0})

print("Encoded class labels:", y_encoded.unique())

# this step converts categorical class labels into numeric labels
# .map() replaces categorical labels using a dictionary

In [ ]:
# -----------------------------
# Step 4: Train / Test Split
# -----------------------------

# Split data into training and test sets
# Stratification preserves class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size = 0.3,
    random_state = 42,
    stratify = y_encoded
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

# stratify = y_encoded -> split the data such that the proportion of each class in y_encoded is preserved in both sets
# it makes sure the training and test sets have the same proportion of malignant and benign cases as the original dataset

In [ ]:
# -----------------------------
# Step 4.1: Check Class Distribution After Stratified Split
# -----------------------------

print("\nClass distribution in the FULL dataset:")
print(y_encoded.value_counts(normalize = True))

print("\nClass distribution in the TRAINING set:")
print(y_train.value_counts(normalize = True))

print("\nClass distribution in the TEST set:")
print(y_test.value_counts(normalize = True))

In [ ]:
# -----------------------------
# Step 5: Feature Scaling
# -----------------------------

# Features have different scales, so normalization is required
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# -----------------------------
# Step 6: Train Logistic Regression Model
# -----------------------------

# Logistic Regression for binary classification
model = LogisticRegression(max_iter = 1000)
model.fit(X_train_scaled, y_train)

In [ ]:
# -----------------------------
# Step 7: Make Predictions
# -----------------------------

# Predicted class labels (0 or 1)
y_pred = model.predict(X_test_scaled)

# Predicted probabilities for the positive class (Malignant = 1)
y_scores = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# -----------------------------
# Step 8: Accuracy and Error Rate on the test set
# -----------------------------

# Accuracy: proportion of correct predictions
accuracy = accuracy_score(y_test, y_pred)

# Error rate: proportion of incorrect predictions
error_rate = 1 - accuracy

print(f"Accuracy (on the test set)   : {accuracy:.4f}")
print(f"Error Rate (on the test set) : {error_rate:.4f}")

In [ ]:
# -----------------------------
# Step 9: Precision, Recall, and F1-score on the test set
# -----------------------------

# Precision: how many predicted malignant cases are actually malignant
precision = precision_score(y_test, y_pred)

# Recall: how many actual malignant cases are correctly detected
recall = recall_score(y_test, y_pred)

# F1-score: harmonic mean of precision and recall
f1 = f1_score(y_test, y_pred)

print(f"Precision (on the test set) : {precision:.4f}")
print(f"Recall (on the test set)    : {recall:.4f}")
print(f"F1-score (on the test set)  : {f1:.4f}")

In [ ]:
# -----------------------------
# Step 10: Confusion Matrix
# -----------------------------

# Confusion matrix summarizes prediction results
cm = confusion_matrix(y_test, y_pred)

# Display confusion matrix with class labels
disp = ConfusionMatrixDisplay(       # creates a ConfusionMatrixDisplay object
    confusion_matrix = cm,           # passes the previously computed confusion matrix (cm)
    display_labels = ["Benign (0)", "Malignant (1)"]   # specifies the labels shown on the axes of the plot
)                                                      # without this, the plot would show only numeric labels (0, 1)

disp.plot(cmap = "Blues")    # calls the plot() method to draw the confusion matrix
                             # Uses a blue color gradient
                             # darker blue = larger number
plt.title("Confusion Matrix (on the test set)")
plt.show()

In [ ]:
# -----------------------------
# Step 11: Precision–Recall Curve
# -----------------------------

# Compute precision and recall values for different thresholds
precision_vals, recall_vals, thresholds = precision_recall_curve(
    y_test,
    y_scores  # predicted probabilities for the positive class
)

# The thresholds used here are the unique predicted probabilities produced by the model, 
# and each one defines a different decision rule
# These thresholds come from y_scores
# For a given threshold t, predict Malignant (1) if predicted_probability ≥ t

# Plot Precision–Recall curve
plt.figure(figsize = (6, 4))
plt.plot(recall_vals, precision_vals, 'b-', linewidth = 2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.grid(True)
plt.show()

# The dominance rule applies in both directions: when precision is the same, higher recall is better; 
# when recall is the same, higher precision is better
# but the PR curve is plotted as the envelope of best precision for each recall
# This PR curve tell us "What precision can I achieve at a given recall?"

In [ ]:
# -----------------------------
# Step 12: Precision–Recall Trade-off vs Threshold
# -----------------------------

# Plot precision and recall as functions of decision threshold
plt.figure(figsize = (6, 4))
plt.plot(thresholds, precision_vals[:-1], 'b--', label = "Precision")
plt.plot(thresholds, recall_vals[:-1], 'r-', label = "Recall")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Precision–Recall Trade-off")
plt.legend()
plt.grid(True)
plt.show()

# This plot shows how precision and recall change as we vary the decision threshold
# As the threshold increases, recall decreases
# As the threshold increases, precision usually increases
# This curve tell us "Which threshold should I choose?"